# Chương 2 (B2B): Phân tích tăng trưởng khu vực và phát hiện bất thường doanh thu Reseller

Notebook được tổ chức theo pipeline:

1. Kiểm tra cấu trúc dữ liệu và phạm vi phân tích Reseller.
2. Tải dữ liệu bán hàng B2B theo grain `Tháng × Khu vực` từ `dwh.fact_reseller_sales`.
3. Kiểm tra mức độ đầy đủ của từng tháng và loại tháng chưa hoàn chỉnh.
4. Tính KPI khu vực B2B, tăng trưởng, khách hàng mới và retention thực tế.
5. Phân tích quy mô – tăng trưởng Reseller.
6. **Data Mining - Phân cụm khu vực B2B bằng K-Means**:
   * Phân cụm trên toàn bộ đặc trưng đã chuẩn hóa; PCA chỉ dùng để trực quan hóa.
   * Tự động chọn số cụm tối ưu K dựa trên Silhouette Score qua nhiều random seeds.
7. Phát hiện tăng/giảm doanh thu bất thường bằng STL.
8. So sánh actual với forecast Reseller (nếu có).
9. Sinh kết luận và gợi ý hành động cụ thể cho kênh B2B.
10. Ghi kết quả vào các bảng riêng cho Reseller trong schema `ml`.

### Câu hỏi kinh doanh cốt lõi (Theo Tài liệu Định hướng Chương 2 - Kênh B2B):
1. **Khu vực Reseller nào đang tăng trưởng và nên đầu tư thêm?**
2. **Khu vực Reseller nào doanh thu thấp nhưng còn dư địa?**
3. **Có sự khác biệt nào về hành vi mua lặp (Retention) của Reseller so với B2C?**
4. **Khu vực Reseller nào suy giảm bất thường về mặt doanh số?**

> **Phạm vi:** Chương này sử dụng `dwh.fact_reseller_sales` và `dwh.dim_product` (để lấy standard_cost tính COGS) nhằm phân tích kênh B2B Reseller.


## 1. Cấu hình và kết nối PostgreSQL DWH

In [ ]:
import sys
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from statsmodels.tsa.seasonal import STL
from sqlalchemy import text

# Cho phép import module trong thư mục src
sys.path.append(os.path.abspath(os.path.join("..")))
from src.common.database import get_dwh_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["figure.figsize"] = (12, 6)

# Tham số có thể điều chỉnh
COMPLETE_MONTH_MIN_RATIO = 1.0
MIN_STL_MONTHS = 24
ANOMALY_Z_THRESHOLD = 2.5
K_MIN = 2
K_MAX = 4
WRITE_TO_DATABASE = False  # Chuyển thành True sau khi kiểm tra kết quả

engine = get_dwh_engine()

def get_table_columns(engine, schema_name: str, table_name: str) -> list[str]:
    query = text(
        '''
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema_name
          AND table_name = :table_name
        ORDER BY ordinal_position
        '''
    )
    with engine.connect() as connection:
        rows = connection.execute(
            query,
            {"schema_name": schema_name, "table_name": table_name},
        ).fetchall()
    return [row[0] for row in rows]

print("Kết nối PostgreSQL DWH thành công.")


## 2. Tải dữ liệu B2B Reseller Sales

In [ ]:
# Truy vấn tổng hợp dữ liệu B2B từ fact_reseller_sales
# Tính toán COGS dựa trên dim_product.standard_cost vì fact_reseller_sales không chứa sẵn cột cost
query = '''
WITH customer_activity AS (
    SELECT DISTINCT
        DATE_TRUNC('month', TO_DATE(CAST(date_key AS VARCHAR), 'YYYYMMDD'))::date AS month_date,
        territory_key,
        reseller_key
    FROM dwh.fact_reseller_sales
    WHERE date_key IS NOT NULL
      AND territory_key IS NOT NULL
      AND reseller_key IS NOT NULL
),
customer_first_purchase AS (
    SELECT
        reseller_key,
        MIN(month_date) AS first_purchase_month
    FROM customer_activity
    GROUP BY reseller_key
),
customer_monthly AS (
    SELECT
        current_activity.month_date,
        current_activity.territory_key,
        COUNT(DISTINCT current_activity.reseller_key) AS active_customers,
        COUNT(
            DISTINCT CASE
                WHEN first_purchase.first_purchase_month = current_activity.month_date
                THEN current_activity.reseller_key
            END
        ) AS new_customers,
        COUNT(
            DISTINCT CASE
                WHEN previous_activity.reseller_key IS NOT NULL
                THEN current_activity.reseller_key
            END
        ) AS retained_customers
    FROM customer_activity AS current_activity
    JOIN customer_first_purchase AS first_purchase
      ON current_activity.reseller_key = first_purchase.reseller_key
    LEFT JOIN customer_activity AS previous_activity
      ON current_activity.reseller_key = previous_activity.reseller_key
     AND current_activity.territory_key = previous_activity.territory_key
     AND previous_activity.month_date =
         (current_activity.month_date - INTERVAL '1 month')::date
    GROUP BY
        current_activity.month_date,
        current_activity.territory_key
),
sales_monthly AS (
    SELECT
        DATE_TRUNC('month', TO_DATE(CAST(sales.date_key AS VARCHAR), 'YYYYMMDD'))::date AS month_date,
        sales.territory_key,
        SUM(COALESCE(sales.line_total, 0)) AS revenue,
        SUM(COALESCE(sales.order_qty, 0) * COALESCE(prod.standard_cost, 0)) AS cogs,
        SUM(COALESCE(sales.line_total, 0) - COALESCE(sales.order_qty, 0) * COALESCE(prod.standard_cost, 0)) AS profit,
        COUNT(DISTINCT sales.sales_order_number) AS orders,
        SUM(COALESCE(sales.order_qty, 0)) AS quantity
    FROM dwh.fact_reseller_sales AS sales
    LEFT JOIN dwh.dim_product AS prod ON sales.product_key = prod.product_key
    WHERE sales.date_key IS NOT NULL
      AND sales.territory_key IS NOT NULL
    GROUP BY
        DATE_TRUNC('month', TO_DATE(CAST(sales.date_key AS VARCHAR), 'YYYYMMDD'))::date,
        sales.territory_key
)
SELECT
    sales.month_date,
    TO_CHAR(sales.month_date, 'YYYYMM') AS month_key,
    sales.territory_key AS territory_id,
    territory.territory_name,
    territory.country_code,
    sales.revenue,
    sales.cogs,
    sales.profit,
    sales.orders,
    sales.quantity,
    COALESCE(customer.active_customers, 0) AS active_customers,
    COALESCE(customer.new_customers, 0) AS new_customers,
    COALESCE(customer.retained_customers, 0) AS retained_customers
FROM sales_monthly AS sales
JOIN dwh.dim_sales_territory AS territory
  ON sales.territory_key = territory.territory_key
LEFT JOIN customer_monthly AS customer
  ON sales.month_date = customer.month_date
 AND sales.territory_key = customer.territory_key
ORDER BY sales.month_date, territory.territory_id
'''

raw_df = pd.read_sql_query(query, engine)
raw_df["month_date"] = pd.to_datetime(raw_df["month_date"])
raw_df["month_key"] = raw_df["month_key"].astype(str)

print(f"Đã tải {len(raw_df):,} dòng dữ liệu B2B.")
display(raw_df.head())


## 3. Kiểm tra mức độ đầy đủ của dữ liệu theo tháng

In [ ]:
expected_territories = raw_df["territory_id"].nunique()

month_coverage = (
    raw_df.groupby(["month_date", "month_key"], as_index=False)
    .agg(
        territory_count=("territory_id", "nunique"),
        monthly_revenue=("revenue", "sum"),
    )
)

month_coverage["coverage_ratio"] = (
    month_coverage["territory_count"] / expected_territories
)
month_coverage["is_complete_month"] = (
    month_coverage["coverage_ratio"] >= COMPLETE_MONTH_MIN_RATIO
)

incomplete_months = month_coverage.loc[
    ~month_coverage["is_complete_month"]
].copy()

print(f"Số khu vực kỳ vọng: {expected_territories}")
print(
    f"Số tháng đầy đủ: {month_coverage['is_complete_month'].sum()} / "
    f"{len(month_coverage)}"
)

if incomplete_months.empty:
    print("Không phát hiện tháng thiếu khu vực.")
else:
    print("Các tháng chưa đầy đủ sẽ bị loại khỏi phần mô hình:")
    display(incomplete_months)

complete_month_keys = set(
    month_coverage.loc[
        month_coverage["is_complete_month"], "month_key"
    ]
)

df = raw_df.loc[raw_df["month_key"].isin(complete_month_keys)].copy()
df = df.sort_values(["territory_id", "month_date"]).reset_index(drop=True)

if df.empty:
    raise RuntimeError(
        "Không còn dữ liệu sau khi áp dụng tiêu chí tháng đầy đủ. "
        "Hãy kiểm tra COMPLETE_MONTH_MIN_RATIO."
    )

print(f"Dữ liệu dùng để phân tích: {len(df):,} dòng.")


## 4. Tính KPI khu vực B2B

In [ ]:
grouped = df.groupby("territory_id", group_keys=False)

df["previous_revenue"] = grouped["revenue"].shift(1)
df["log_growth"] = grouped["revenue"].transform(
    lambda series: np.log1p(series).diff()
)
df["yoy_growth"] = grouped["revenue"].pct_change(12)

df["previous_active_customers"] = grouped["active_customers"].shift(1)
df["existing_customers"] = (
    df["active_customers"] - df["new_customers"]
).clip(lower=0)

df["retention_rate"] = np.where(
    df["previous_active_customers"] > 0,
    df["retained_customers"] / df["previous_active_customers"],
    np.nan,
)

df["churned_customers"] = np.where(
    df["previous_active_customers"].notna(),
    (
        df["previous_active_customers"] - df["retained_customers"]
    ).clip(lower=0),
    np.nan,
)

df["profit_margin"] = np.where(
    df["revenue"] != 0,
    df["profit"] / df["revenue"],
    np.nan,
)

display(
    df[
        [
            "month_key",
            "territory_name",
            "revenue",
            "profit_margin",
            "active_customers",
            "new_customers",
            "retained_customers",
            "retention_rate",
            "log_growth",
            "yoy_growth",
        ]
    ].head(15)
)


## 5. EDA: xu hướng và ma trận quy mô – tăng trưởng Reseller

In [ ]:
revenue_pivot = df.pivot(
    index="month_date",
    columns="territory_name",
    values="revenue",
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x="month_date", y="revenue", hue="territory_name", marker="o", alpha=0.75)
plt.title("Xu hướng doanh thu B2B Reseller theo tháng và khu vực")
plt.xlabel("Tháng")
plt.ylabel("Doanh thu (USD)")
plt.legend(
    title="Khu vực",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.tight_layout()
plt.show()


In [ ]:
territory_summary = (
    df.groupby(["territory_id", "territory_name"], as_index=False)
    .agg(
        avg_revenue=("revenue", "mean"),
        median_log_growth=("log_growth", "median"),
        median_yoy_growth=("yoy_growth", "median"),
        total_profit=("profit", "sum"),
        mean_profit_margin=("profit_margin", "mean"),
        avg_active_customers=("active_customers", "mean"),
        mean_retention_rate=("retention_rate", "mean"),
        revenue_std=("revenue", "std"),
    )
)

territory_summary["monthly_growth_equivalent"] = np.expm1(
    territory_summary["median_log_growth"]
)
territory_summary["revenue_cv"] = np.where(
    territory_summary["avg_revenue"] != 0,
    territory_summary["revenue_std"]
    / territory_summary["avg_revenue"].abs(),
    np.nan,
)

fig, ax = plt.subplots(figsize=(13, 7))
sns.scatterplot(
    data=territory_summary,
    x="monthly_growth_equivalent",
    y="avg_revenue",
    hue="territory_name",
    size="avg_active_customers",
    sizes=(50, 450),
    alpha=0.85,
    edgecolor="white",
    linewidth=1.5,
    ax=ax
)

# Định dạng các trục hiển thị trực quan đẹp mắt
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f"${x:,.0f}"))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Lấy đường phân chia góc phần tư (trung vị)
x_mid = territory_summary["monthly_growth_equivalent"].median()
y_mid = territory_summary["avg_revenue"].median()

ax.axvline(x_mid, color="red", linestyle="--", linewidth=1.2, alpha=0.6)
ax.axhline(y_mid, color="red", linestyle="--", linewidth=1.2, alpha=0.6)

# Chú thích nhãn phân khu kinh doanh cho 4 góc phần tư
x_lim = ax.get_xlim()
y_lim = ax.get_ylim()

ax.text(x_mid + (x_lim[1] - x_mid)*0.5, y_mid + (y_lim[1] - y_mid)*0.7, 
        "NGÔI SAO\n(Quy Mô & Tăng Trưởng Cao)", color="green", ha="center", va="center", weight="bold", fontsize=10, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.3'))
ax.text(x_lim[0] + (x_mid - x_lim[0])*0.5, y_mid + (y_lim[1] - y_mid)*0.7, 
        "TRỤ CỘT\n(Quy Mô Lớn, Tăng Trưởng Ổn Định)", color="blue", ha="center", va="center", weight="bold", fontsize=10, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.3'))
ax.text(x_mid + (x_lim[1] - x_mid)*0.5, y_lim[0] + (y_mid - y_lim[0])*0.3, 
        "TIỀM NĂNG\n(Quy Mô Nhỏ, Tăng Trưởng Nhanh)", color="orange", ha="center", va="center", weight="bold", fontsize=10, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.3'))
ax.text(x_lim[0] + (x_mid - x_lim[0])*0.5, y_lim[0] + (y_mid - y_lim[0])*0.3, 
        "CẦN CẢI THIỆN\n(Quy Mô & Tăng Trưởng Thấp)", color="red", ha="center", va="center", weight="bold", fontsize=10, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.3'))

plt.title("Ma trận Quy mô vs Tăng trưởng Doanh thu B2B Reseller", fontsize=14, weight="bold", pad=15)
plt.xlabel("Tốc độ tăng trưởng tháng đại diện (Monthly Growth Equivalent)", fontsize=11)
plt.ylabel("Doanh thu trung bình mỗi tháng (USD)", fontsize=11)
plt.legend(
    title="Lãnh thổ",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.tight_layout()
plt.show()

display(
    territory_summary.sort_values("avg_revenue", ascending=False)
)


### Phân tích Khám phá & Đánh giá Ma trận Quy mô – Tăng trưởng B2B

Dựa trên biểu đồ ma trận Quy mô vs Tăng trưởng doanh thu B2B Reseller phía trên, chúng ta có thể bước đầu trả lời các câu hỏi kinh doanh cốt lõi của kênh B2B:

* **Câu hỏi 1: Khu vực Reseller nào đang tăng trưởng tốt và nên đầu tư thêm?**
  * **Southwest** ($512K/tháng) và **Canada** ($399K/tháng) là hai thị trường cột trụ quy mô lớn nhất đối với kênh Reseller, với đà tăng trưởng dương nhẹ (Monthly Growth Equivalent đạt từ 0.5% đến 2.1%). Đây là những thị trường đã định hình mạng lưới nhà phân phối vững chắc và tiếp tục mang lại phần lớn doanh thu B2B.

* **Câu hỏi 2: Khu vực Reseller nào doanh thu thấp nhưng còn nhiều dư địa?**
  * **Australia** có doanh thu trung bình tháng B2B thấp nhất ($132K), cho thấy kênh nhà phân phối tại đây chưa phát triển mạnh mẽ so với kênh bán lẻ B2C (vốn là thị trường chủ lực ở B2C). Điều này chỉ ra dư địa rất lớn cho việc ký kết thêm các hợp đồng Reseller mới tại Úc.
  * **Cảnh báo cực kỳ nghiêm trọng về khả năng sinh lời gộp**: Khác hoàn toàn với B2C, tất cả các khu vực kinh doanh B2B của AdventureWorks đều đang **hoạt động dưới mức hòa vốn (Biên lợi nhuận gộp âm từ -1.7% đến -8.0%)**. Tổng doanh thu B2B Reseller đạt $80.48M trong khi COGS sản xuất lên tới $82.80M, tạo ra khoản lỗ gộp lũy kế **-$2.31M**. Điều này chỉ ra chính sách chiết khấu thương mại (reseller discount) cho các đối tác B2B đang bị lạm dụng vượt mức chi phí sản xuất tiêu chuẩn, cần rà soát lại gấp.

## 6. Phân cụm khu vực B2B bằng K-Means

In [ ]:
features_df = territory_summary[
    [
        "territory_id",
        "territory_name",
        "avg_revenue",
        "median_log_growth",
        "revenue_cv",
        "avg_active_customers",
        "mean_profit_margin",
        "mean_retention_rate",
    ]
].copy()

feature_columns = [
    "avg_revenue",
    "median_log_growth",
    "revenue_cv",
    "avg_active_customers",
    "mean_profit_margin",
    "mean_retention_rate",
]

features_df[feature_columns] = (
    features_df[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

for column in feature_columns:
    features_df[column] = features_df[column].fillna(
        features_df[column].median()
    )

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df[feature_columns])

candidate_k_values = range(
    K_MIN,
    min(K_MAX, len(features_df) - 1) + 1,
)
random_seeds = [0, 7, 21, 42, 84]

k_evaluation_rows = []

for k in candidate_k_values:
    for seed in random_seeds:
        model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=20,
        )
        labels = model.fit_predict(scaled_features)
        k_evaluation_rows.append(
            {
                "k": k,
                "seed": seed,
                "silhouette_score": silhouette_score(
                    scaled_features,
                    labels,
                ),
                "inertia": model.inertia_,
            }
        )

k_evaluation = pd.DataFrame(k_evaluation_rows)

k_summary = (
    k_evaluation.groupby("k", as_index=False)
    .agg(
        mean_silhouette=("silhouette_score", "mean"),
        std_silhouette=("silhouette_score", "std"),
        mean_inertia=("inertia", "mean"),
    )
    .sort_values(
        ["mean_silhouette", "std_silhouette"],
        ascending=[False, True],
    )
)

best_k = int(k_summary.iloc[0]["k"])
best_silhouette = float(k_summary.iloc[0]["mean_silhouette"])

print(
    f"K được chọn: {best_k}; "
    f"Silhouette trung bình: {best_silhouette:.4f}"
)
display(k_summary)


In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50,
)
features_df["cluster_label"] = kmeans.fit_predict(scaled_features)

# PCA chỉ dùng để trực quan hóa
pca = PCA(n_components=2, random_state=42)
pca_coordinates = pca.fit_transform(scaled_features)
features_df["pca_1"] = pca_coordinates[:, 0]
features_df["pca_2"] = pca_coordinates[:, 1]

cluster_profile = (
    features_df.groupby("cluster_label", as_index=False)
    .agg(
        territory_count=("territory_id", "count"),
        avg_revenue=("avg_revenue", "mean"),
        median_log_growth=("median_log_growth", "mean"),
        revenue_cv=("revenue_cv", "mean"),
        avg_active_customers=("avg_active_customers", "mean"),
        mean_profit_margin=("mean_profit_margin", "mean"),
        mean_retention_rate=("mean_retention_rate", "mean"),
    )
)

# Điểm chiến lược dùng để gán nhãn tương đối, không thay thế đánh giá nghiệp vụ
cluster_profile["scale_score"] = (
    cluster_profile["avg_revenue"].rank(pct=True)
    + cluster_profile["avg_active_customers"].rank(pct=True)
    + cluster_profile["mean_profit_margin"].rank(pct=True)
) / 3

cluster_profile["growth_score"] = (
    cluster_profile["median_log_growth"].rank(pct=True)
    + cluster_profile["mean_retention_rate"].rank(pct=True)
) / 2

cluster_profile["risk_score"] = (
    cluster_profile["revenue_cv"].rank(pct=True)
    + (-cluster_profile["mean_profit_margin"]).rank(pct=True)
) / 2

cluster_profile["strategic_score"] = (
    0.45 * cluster_profile["scale_score"]
    + 0.35 * cluster_profile["growth_score"]
    - 0.20 * cluster_profile["risk_score"]
)

ordered_clusters = (
    cluster_profile.sort_values(
        "strategic_score",
        ascending=False,
    )["cluster_label"]
    .tolist()
)

label_sets = {
    2: [
        "Thị trường chủ lực",
        "Thị trường cần cải thiện",
    ],
    3: [
        "Thị trường chủ lực",
        "Thị trường tăng trưởng tiềm năng",
        "Thị trường cần cải thiện",
    ],
    4: [
        "Thị trường chủ lực",
        "Thị trường tăng trưởng tiềm năng",
        "Thị trường ổn định quy mô trung bình",
        "Thị trường cần cải thiện",
    ],
}

cluster_name_map = {
    cluster_id: label_sets[best_k][position]
    for position, cluster_id in enumerate(ordered_clusters)
}

features_df["cluster_name"] = features_df["cluster_label"].map(
    cluster_name_map
)
cluster_profile["cluster_name"] = cluster_profile[
    "cluster_label"
].map(cluster_name_map)

cluster_members = (
    features_df.groupby(
        ["cluster_label", "cluster_name"],
        as_index=False,
    )
    .agg(
        territories=(
            "territory_name",
            lambda values: ", ".join(sorted(values)),
        )
    )
)

cluster_profile = cluster_profile.merge(
    cluster_members,
    on=["cluster_label", "cluster_name"],
    how="left",
)

print(
    "Tỷ lệ phương sai được giải thích bởi PCA 2D:",
    pca.explained_variance_ratio_.sum(),
)
display(
    cluster_profile.sort_values(
        "strategic_score",
        ascending=False,
    )
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# 1. Biểu đồ A: PCA 2D (Kèm Biplot Loading Vectors)
sns.scatterplot(
    data=features_df,
    x="pca_1",
    y="pca_2",
    hue="cluster_name",
    s=200,
    alpha=0.85,
    edgecolor="white",
    linewidth=1.5,
    ax=axes[0]
)

for _, row in features_df.iterrows():
    axes[0].annotate(
        row["territory_name"],
        (row["pca_1"], row["pca_2"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
        weight="bold"
    )

# Vẽ các vector loadings của 5 đặc trưng gốc
friendly_names = {
    "avg_revenue": "Doanh thu TB",
    "median_log_growth": "Tăng trưởng TB",
    "revenue_cv": "Độ biến động",
    "avg_active_customers": "Khách hàng TB",
    "mean_profit_margin": "Biên LN TB",
    "mean_retention_rate": "Retention TB"
}

scale_factor = 4.0  # Tỷ lệ phóng lớn vector trên biểu đồ PCA
for i, feature in enumerate(feature_columns):
    arrow_x = pca.components_[0, i] * scale_factor
    arrow_y = pca.components_[1, i] * scale_factor
    
    # Vẽ mũi tên
    axes[0].arrow(
        0, 0, arrow_x, arrow_y,
        color="red", alpha=0.7, linewidth=1.5,
        head_width=0.12, head_length=0.18
    )
    # Ghi nhãn
    label = friendly_names.get(feature, feature)
    axes[0].text(
        arrow_x * 1.15, arrow_y * 1.15,
        label, color="darkred", ha="center", va="center",
        fontsize=10, weight="bold"
    )

axes[0].set_title("A. Kết quả phân cụm trong không gian PCA 2D (Kèm Biplot Loading Vectors)", fontsize=13, weight="bold", pad=12)
axes[0].set_xlabel("Principal Component 1 (PCA 1)", fontsize=11)
axes[0].set_ylabel("Principal Component 2 (PCA 2)", fontsize=11)
axes[0].legend(title="Phân khúc", loc="upper left")

# 2. Biểu đồ B: Quy mô vs Tăng trưởng thực tế có vẽ phân cụm
sns.scatterplot(
    data=features_df,
    x="median_log_growth",
    y="avg_revenue",
    hue="cluster_name",
    size="avg_active_customers",
    sizes=(50, 400),
    alpha=0.85,
    edgecolor="white",
    linewidth=1.5,
    ax=axes[1]
)

for _, row in features_df.iterrows():
    axes[1].annotate(
        row["territory_name"],
        (row["median_log_growth"], row["avg_revenue"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
        weight="bold"
    )

# Định dạng các trục kinh doanh thực tế
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f"${x:,.0f}"))
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Đường trung vị phân cụm
x_mid_b = features_df["median_log_growth"].median()
y_mid_b = features_df["avg_revenue"].median()
axes[1].axvline(x_mid_b, color="gray", linestyle="--", linewidth=1)
axes[1].axhline(y_mid_b, color="gray", linestyle="--", linewidth=1)

axes[1].set_title("B. Phân khúc trên trục Tăng trưởng vs Doanh thu thực tế", fontsize=13, weight="bold", pad=12)
axes[1].set_xlabel("Tốc độ tăng trưởng trung vị (Median log-growth)", fontsize=11)
axes[1].set_ylabel("Doanh thu trung bình mỗi tháng (USD)", fontsize=11)
axes[1].legend(title="Phân khúc", loc="upper right")

plt.tight_layout()
plt.show()


### Phân tích Phân cụm & Đánh giá Sự khác biệt Hành vi mua lặp (Clustering & B2B Retention)

Kết quả phân cụm B2B chỉ ra những insight kinh doanh cực kỳ độc đáo và khác biệt hoàn toàn với B2C:

* **Câu hỏi 3: Có sự khác biệt nào về hành vi mua lặp (Retention) của Reseller so với B2C?**
  * **Trả lời**: Tỷ lệ giữ chân khách hàng hàng tháng (Monthly Retention Rate) của tất cả 10 khu vực trong kênh Reseller đều bằng **0.0%**. 
  * **Lý do chuyên môn**: Đây là đặc tính cơ bản của mô hình B2B. Các Reseller thường mua hàng với số lượng lớn (bulk purchase) để lưu kho và phân phối dần, do đó tần suất đặt hàng của họ thường là **hàng quý hoặc hàng năm** thay vì phát sinh giao dịch đều đặn mỗi tháng như khách hàng cá nhân B2C. 
  * **Đề xuất kỹ thuật**: Chỉ số retention rate theo tháng không phản ánh đúng lòng trung thành của Reseller. AdventureWorks cần chuyển sang sử dụng chu kỳ quan sát rộng hơn (Quarterly/Annual Observation Window) để đánh giá sức khỏe tệp đối tác B2B.

* **Đặc trưng các phân cụm B2B (K=4)**:
  * **Thị trường chủ lực (Cluster 1)**: Southwest, Northwest và Canada. Quy mô doanh thu rất lớn ($345K - $512K/tháng) và số lượng reseller active đông đảo nhất (đại diện cho lõi doanh số B2B của công ty).
  * **Thị trường ổn định quy mô trung bình (Cluster 0)**: Central, Northeast, Southeast, Germany, và United Kingdom. Quy mô doanh số vừa phải ($168K - $219K/tháng), tăng trưởng đi ngang.
  * **Thị trường tăng trưởng tiềm năng (Cluster 2)**: France. Mức doanh thu khá và có sự cải thiện tốt về mặt tăng trưởng trong thời gian gần đây.
  * **Thị trường cần cải thiện (Cluster 3)**: Australia. Đây là thị trường hoạt động kém nhất đối với Reseller với doanh số trung bình cực kỳ thấp ($132K) và biên lợi nhuận gộp âm sâu nhất (-8.0%).

## 7. Phát hiện bất thường bằng STL

In [ ]:
def robust_zscore(values: pd.Series) -> pd.Series:
    median_value = values.median()
    mad = np.median(np.abs(values - median_value))

    if mad == 0 or np.isnan(mad):
        std = values.std(ddof=0)
        if std == 0 or np.isnan(std):
            return pd.Series(
                np.zeros(len(values)),
                index=values.index,
            )
        return (values - values.mean()) / std

    return 0.6745 * (values - median_value) / mad


anomaly_frames = []
skipped_territories = []

for territory_id, territory_data in df.groupby("territory_id"):
    series_df = (
        territory_data[
            ["month_date", "month_key", "territory_id", "territory_name", "revenue"]
        ]
        .drop_duplicates(subset=["month_date"])
        .set_index("month_date")
        .sort_index()
    )

    monthly_series = series_df["revenue"].asfreq("MS")

    if monthly_series.isna().any():
        skipped_territories.append(
            {
                "territory_id": territory_id,
                "territory_name": territory_data["territory_name"].iloc[0],
                "reason": "Chuỗi còn thiếu tháng sau khi asfreq('MS')",
            }
        )
        continue

    if len(monthly_series) < MIN_STL_MONTHS:
        skipped_territories.append(
            {
                "territory_id": territory_id,
                "territory_name": territory_data["territory_name"].iloc[0],
                "reason": (
                    f"Chỉ có {len(monthly_series)} tháng; "
                    f"yêu cầu tối thiểu {MIN_STL_MONTHS}"
                ),
            }
        )
        continue

    stl_result = STL(
        monthly_series,
        period=12,
        robust=True,
    ).fit()

    residual_z = robust_zscore(stl_result.resid)
    expected_revenue = stl_result.trend + stl_result.seasonal

    territory_anomaly = pd.DataFrame(
        {
            "month_date": monthly_series.index,
            "territory_id": territory_id,
            "actual_revenue": monthly_series.values,
            "expected_revenue_stl": expected_revenue.values,
            "stl_residual": stl_result.resid.values,
            "residual_z": residual_z.values,
        }
    )

    territory_anomaly["is_anomaly"] = (
        territory_anomaly["residual_z"].abs()
        >= ANOMALY_Z_THRESHOLD
    )

    territory_anomaly["anomaly_type"] = np.select(
        [
            territory_anomaly["residual_z"]
            <= -ANOMALY_Z_THRESHOLD,
            territory_anomaly["residual_z"]
            >= ANOMALY_Z_THRESHOLD,
        ],
        [
            "Abnormal Decline",
            "Abnormal Increase",
        ],
        default="Normal",
    )

    territory_anomaly["month_key"] = (
        territory_anomaly["month_date"]
        .dt.strftime("%Y%m")
    )
    anomaly_frames.append(territory_anomaly)

if anomaly_frames:
    anomaly_df = pd.concat(
        anomaly_frames,
        ignore_index=True,
    )
else:
    anomaly_df = pd.DataFrame(
        columns=[
            "month_date",
            "month_key",
            "territory_id",
            "actual_revenue",
            "expected_revenue_stl",
            "stl_residual",
            "residual_z",
            "is_anomaly",
            "anomaly_type",
        ]
    )

if skipped_territories:
    print("Các khu vực không đủ điều kiện chạy STL:")
    display(pd.DataFrame(skipped_territories))

anomaly_summary = (
    anomaly_df.loc[anomaly_df["is_anomaly"]]
    .groupby(
        ["territory_id", "anomaly_type"],
        as_index=False,
    )
    .agg(
        anomaly_count=("month_key", "count"),
        largest_abs_z=("residual_z", lambda values: values.abs().max()),
    )
)

print(
    f"Tổng số tháng bất thường B2B: "
    f"{int(anomaly_df['is_anomaly'].sum()) if not anomaly_df.empty else 0}"
)
display(anomaly_summary)


In [ ]:
# Trực quan hóa các khu vực có nhiều lần giảm bất thường nhất
decline_counts = (
    anomaly_df.loc[
        anomaly_df["anomaly_type"] == "Abnormal Decline"
    ]
    .groupby("territory_id")
    .size()
    .sort_values(ascending=False)
)

top_decline_territories = decline_counts.head(3).index.tolist()

for territory_id in top_decline_territories:
    territory_name = df.loc[
        df["territory_id"] == territory_id,
        "territory_name",
    ].iloc[0]

    plot_df = anomaly_df.loc[
        anomaly_df["territory_id"] == territory_id
    ].copy()

    plt.figure(figsize=(12, 5))
    sns.lineplot(
        data=plot_df,
        x="month_date",
        y="actual_revenue",
        marker="o",
        label="Actual Revenue",
    )
    sns.lineplot(
        data=plot_df,
        x="month_date",
        y="expected_revenue_stl",
        linestyle="--",
        label="STL Expected Revenue",
    )

    decline_points = plot_df.loc[
        plot_df["anomaly_type"] == "Abnormal Decline"
    ]

    if not decline_points.empty:
        sns.scatterplot(
            data=decline_points,
            x="month_date",
            y="actual_revenue",
            s=90,
            color="red",
            label="Abnormal Decline",
            zorder=3,
        )

    plt.title(f"Biến động doanh thu bất thường B2B – {territory_name}")
    plt.xlabel("Tháng")
    plt.ylabel("Doanh thu (USD)")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. So sánh actual với forecast

In [ ]:
forecast_table = "reseller_forecast_predictions"
forecast_columns = get_table_columns(
    engine,
    "ml",
    forecast_table,
)

if forecast_columns:
    # Phần này sẽ chạy nếu có bảng dự báo B2B được chuẩn bị sẵn
    pass
else:
    print(
        "Chưa tìm thấy bảng ml.reseller_forecast_predictions. "
        "Phần forecast B2B được bỏ qua."
    )
    forecast_evaluation = pd.DataFrame()


### Phân tích Biến động Bất thường doanh thu Reseller

* **Câu hỏi 4: Khu vực Reseller nào suy giảm bất thường về mặt doanh số?**
  * Mô hình STL robust lọc tách thành công các tháng sụt giảm cực đoan. Phân tích ghi nhận **Germany** là khu vực có tính bất ổn định giao dịch B2B cao nhất. 
  * Giao dịch B2B của các Reseller có biên độ lớn, do đó một đơn hàng bị hủy hoặc hoãn giao từ một nhà phân phối lớn sẽ lập tức tạo ra điểm gãy doanh thu bất thường ("Abnormal Decline") trên đồ thị lịch sử.

## 9. Tổng hợp insight và gợi ý hành động

In [ ]:
decline_count_by_territory = (
    anomaly_df.loc[
        anomaly_df["anomaly_type"] == "Abnormal Decline"
    ]
    .groupby("territory_id")
    .size()
    .rename("abnormal_decline_count")
)

action_df = features_df.merge(
    decline_count_by_territory,
    on="territory_id",
    how="left",
)

action_df["abnormal_decline_count"] = (
    action_df["abnormal_decline_count"]
    .fillna(0)
    .astype(int)
)

revenue_median = action_df["avg_revenue"].median()
growth_median = action_df["median_log_growth"].median()
retention_median = action_df["mean_retention_rate"].median()


def recommend_action(row: pd.Series) -> str:
    recommendations = []

    if row["mean_profit_margin"] < 0:
        recommendations.append(
            "Khẩn cấp tái định cấu trúc chiết khấu thương mại (Discount)"
        )
    elif (
        row["avg_revenue"] < revenue_median
        and row["median_log_growth"] > growth_median
    ):
        recommendations.append(
            "Tìm kiếm và phát triển thêm nhà phân phối mới"
        )
    else:
        recommendations.append(
            "Duy trì quan hệ đối tác chiến lược và tối ưu chuỗi cung ứng"
        )

    if row["abnormal_decline_count"] > 0:
        recommendations.append(
            "Kiểm tra tiến độ thực hiện hợp đồng đại lý"
        )

    return "; ".join(recommendations)


action_df["recommended_action"] = action_df.apply(
    recommend_action,
    axis=1,
)

action_columns = [
    "territory_name",
    "cluster_name",
    "avg_revenue",
    "median_log_growth",
    "mean_profit_margin",
    "mean_retention_rate",
    "abnormal_decline_count",
    "recommended_action",
]

display(
    action_df[action_columns]
    .sort_values("avg_revenue", ascending=False)
)


In [ ]:
top_revenue_row = action_df.loc[
    action_df["avg_revenue"].idxmax()
]
top_growth_row = action_df.loc[
    action_df["median_log_growth"].idxmax()
]
lowest_margin_row = action_df.loc[
    action_df["mean_profit_margin"].idxmin()
]

print("=== KẾT LUẬN TÓM TẮT CHƯƠNG 2 B2B ===")
print(
    f"- Khu vực Reseller có quy mô doanh thu trung bình lớn nhất: "
    f"{top_revenue_row['territory_name']}."
)
print(
    f"- Khu vực Reseller có median log-growth cao nhất: "
    f"{top_growth_row['territory_name']}."
)
print(
    f"- Khu vực Reseller có biên lợi nhuận trung bình thấp nhất: "
    f"{lowest_margin_row['territory_name']}."
)


### KẾT LUẬN CHIẾN LƯỢC & KHUYẾN NGHỊ HÀNH ĐỘNG KÊNH B2B

#### 1. Khủng hoảng Biên lợi nhuận gộp âm diện rộng
* **Thực trạng**: Đây là phát hiện đáng báo động nhất của kênh B2B. Toàn bộ 10 khu vực bán hàng qua Reseller đều ghi nhận biên lợi nhuận gộp âm. Điều này chứng tỏ AdventureWorks đang bán sỉ dưới giá vốn sản xuất thực tế sau khi đã trừ đi các khoản chiết khấu lớn (`unit_price_discount`).
* **Khuyến nghị khẩn cấp**: 
  * **Cắt giảm tỷ lệ chiết khấu tối đa**: Thiết lập lại trần chiết khấu (discount ceiling) cho từng dòng sản phẩm. Không ký kết các hợp đồng đại lý có mức đơn giá sau chiết khấu thấp hơn `standard_cost`.
  * **Đàm phán tăng giá sỉ**: Thực hiện tăng giá bán sỉ tối thiểu 5% đối với các đối tác đại lý lớn tại **Southwest** và **Canada** để đưa biên lợi nhuận gộp trở lại mức dương.

#### 2. Chiến lược phát triển Đại lý theo vùng địa lý
* **Australia**: Mặc dù là thị trường B2C rất tiềm năng, kênh Reseller B2B tại Úc đang hoạt động tệ nhất ($132K/tháng). Cần cử đội ngũ phát triển kinh doanh sang thiết lập quan hệ và ký kết hợp đồng phân phối mới với các chuỗi cửa hàng xe đạp lớn tại đây.
* **Southwest & Northwest**: Đây là các lõi doanh số B2B. Dù đang lỗ gộp nhẹ (~ -3.5%), quy mô doanh số lớn giúp gánh chi phí cố định cho nhà máy. Cần ưu tiên tối ưu hóa chuỗi cung ứng và logictics nội địa Bắc Mỹ để giảm chi phí COGS vận chuyển cho nhóm này.

#### 3. Chuyển đổi chỉ số đo lường Retention
* **Hành động**: Dừng sử dụng chỉ số Retention rate theo tháng trên báo cáo BI/Superset dành cho đối tượng Reseller. Chuyển sang sử dụng chu kỳ đo lường theo **Quý (Quarterly Retention Rate)** để phản ánh đúng chu kỳ tái đặt hàng của doanh nghiệp.

## 10. Ghi kết quả vào PostgreSQL phục vụ Superset (Reseller Tables)

Kết quả B2B được ghi vào các bảng riêng để tránh ghi đè lên dữ liệu B2C:

- `ml.reseller_territory_monthly_metrics`
- `ml.reseller_territory_cluster_result`
- `ml.reseller_territory_anomaly_monthly`
- `ml.reseller_territory_forecast_evaluation`
- `ml.vw_reseller_territory_analysis_dashboard` (View tổng hợp B2B cho Superset)


In [ ]:
analysis_run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)
analysis_timestamp = datetime.now(timezone.utc)

monthly_output = df[
    [
        "month_key",
        "month_date",
        "territory_id",
        "territory_name",
        "country_code",
        "revenue",
        "cogs",
        "profit",
        "profit_margin",
        "orders",
        "quantity",
        "active_customers",
        "new_customers",
        "existing_customers",
        "retained_customers",
        "previous_active_customers",
        "retention_rate",
        "churned_customers",
        "log_growth",
        "yoy_growth",
    ]
].copy()
monthly_output["analysis_run_id"] = analysis_run_id
monthly_output["analyzed_at"] = analysis_timestamp

cluster_output = features_df[
    [
        "territory_id",
        "territory_name",
        "cluster_label",
        "cluster_name",
        "avg_revenue",
        "median_log_growth",
        "revenue_cv",
        "avg_active_customers",
        "mean_profit_margin",
        "mean_retention_rate",
        "pca_1",
        "pca_2",
    ]
].copy()
cluster_output["best_k"] = best_k
cluster_output["mean_silhouette"] = best_silhouette
cluster_output["analysis_run_id"] = analysis_run_id
cluster_output["analyzed_at"] = analysis_timestamp

anomaly_output = anomaly_df.copy()
if not anomaly_output.empty:
    anomaly_output["analysis_run_id"] = analysis_run_id
    anomaly_output["analyzed_at"] = analysis_timestamp

forecast_output = forecast_evaluation.copy()
if not forecast_output.empty:
    forecast_output["analysis_run_id"] = analysis_run_id
    forecast_output["analyzed_at"] = analysis_timestamp

print(
    {
        "monthly_rows": len(monthly_output),
        "cluster_rows": len(cluster_output),
        "anomaly_rows": len(anomaly_output),
        "forecast_rows": len(forecast_output),
        "write_enabled": WRITE_TO_DATABASE,
    }
)


In [ ]:
if WRITE_TO_DATABASE:
    ddl_statements = [
        "CREATE SCHEMA IF NOT EXISTS ml",
        '''
        CREATE TABLE IF NOT EXISTS ml.reseller_territory_monthly_metrics (
            month_key VARCHAR(6) NOT NULL,
            month_date DATE NOT NULL,
            territory_id INTEGER NOT NULL,
            territory_name TEXT NOT NULL,
            country_code TEXT,
            revenue DOUBLE PRECISION,
            cogs DOUBLE PRECISION,
            profit DOUBLE PRECISION,
            profit_margin DOUBLE PRECISION,
            orders BIGINT,
            quantity DOUBLE PRECISION,
            active_customers BIGINT,
            new_customers BIGINT,
            existing_customers BIGINT,
            retained_customers BIGINT,
            previous_active_customers DOUBLE PRECISION,
            retention_rate DOUBLE PRECISION,
            churned_customers DOUBLE PRECISION,
            log_growth DOUBLE PRECISION,
            yoy_growth DOUBLE PRECISION,
            analysis_run_id TEXT NOT NULL,
            analyzed_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (month_key, territory_id)
        )
        ''',
        '''
        CREATE TABLE IF NOT EXISTS ml.reseller_territory_cluster_result (
            territory_id INTEGER PRIMARY KEY,
            territory_name TEXT NOT NULL,
            cluster_label INTEGER NOT NULL,
            cluster_name TEXT NOT NULL,
            avg_revenue DOUBLE PRECISION,
            median_log_growth DOUBLE PRECISION,
            revenue_cv DOUBLE PRECISION,
            avg_active_customers DOUBLE PRECISION,
            mean_profit_margin DOUBLE PRECISION,
            mean_retention_rate DOUBLE PRECISION,
            pca_1 DOUBLE PRECISION,
            pca_2 DOUBLE PRECISION,
            best_k INTEGER,
            mean_silhouette DOUBLE PRECISION,
            analysis_run_id TEXT NOT NULL,
            analyzed_at TIMESTAMPTZ NOT NULL
        )
        ''',
        '''
        CREATE TABLE IF NOT EXISTS ml.reseller_territory_anomaly_monthly (
            month_date DATE NOT NULL,
            month_key VARCHAR(6) NOT NULL,
            territory_id INTEGER NOT NULL,
            actual_revenue DOUBLE PRECISION,
            expected_revenue_stl DOUBLE PRECISION,
            stl_residual DOUBLE PRECISION,
            residual_z DOUBLE PRECISION,
            is_anomaly BOOLEAN,
            anomaly_type TEXT,
            analysis_run_id TEXT NOT NULL,
            analyzed_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (month_key, territory_id)
        )
        ''',
        '''
        CREATE TABLE IF NOT EXISTS ml.reseller_territory_forecast_evaluation (
            month_key VARCHAR(6) NOT NULL,
            month_date DATE NOT NULL,
            territory_id INTEGER NOT NULL,
            territory_name TEXT,
            revenue DOUBLE PRECISION,
            predicted_revenue DOUBLE PRECISION,
            forecast_error DOUBLE PRECISION,
            absolute_error DOUBLE PRECISION,
            forecast_error_pct DOUBLE PRECISION,
            below_expected_flag BOOLEAN,
            model_version TEXT,
            predicted_at TIMESTAMPTZ,
            created_at TIMESTAMPTZ,
            forecast_horizon INTEGER,
            analysis_run_id TEXT NOT NULL,
            analyzed_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (month_key, territory_id)
        )
        ''',
    ]

    with engine.begin() as connection:
        for statement in ddl_statements:
            connection.execute(text(statement))

        connection.execute(
            text("TRUNCATE TABLE ml.reseller_territory_monthly_metrics")
        )
        connection.execute(
            text("TRUNCATE TABLE ml.reseller_territory_cluster_result")
        )
        connection.execute(
            text("TRUNCATE TABLE ml.reseller_territory_anomaly_monthly")
        )
        connection.execute(
            text("TRUNCATE TABLE ml.reseller_territory_forecast_evaluation")
        )

    monthly_output.to_sql(
        "reseller_territory_monthly_metrics",
        engine,
        schema="ml",
        if_exists="append",
        index=False,
        method="multi",
    )

    cluster_output.to_sql(
        "reseller_territory_cluster_result",
        engine,
        schema="ml",
        if_exists="append",
        index=False,
        method="multi",
    )

    if not anomaly_output.empty:
        anomaly_output.to_sql(
            "reseller_territory_anomaly_monthly",
            engine,
            schema="ml",
            if_exists="append",
            index=False,
            method="multi",
        )

    if not forecast_output.empty:
        allowed_forecast_columns = [
            "month_key",
            "month_date",
            "territory_id",
            "territory_name",
            "revenue",
            "predicted_revenue",
            "forecast_error",
            "absolute_error",
            "forecast_error_pct",
            "below_expected_flag",
            "model_version",
            "predicted_at",
            "created_at",
            "forecast_horizon",
            "analysis_run_id",
            "analyzed_at",
        ]

        for optional_column in [
            "model_version",
            "predicted_at",
            "created_at",
            "forecast_horizon",
        ]:
            if optional_column not in forecast_output.columns:
                forecast_output[optional_column] = None

        forecast_output[
            allowed_forecast_columns
        ].to_sql(
            "reseller_territory_forecast_evaluation",
            engine,
            schema="ml",
            if_exists="append",
            index=False,
            method="multi",
        )

    dashboard_view_sql = '''
    CREATE OR REPLACE VIEW ml.vw_reseller_territory_analysis_dashboard AS
    SELECT
        monthly.*,
        cluster.cluster_label,
        cluster.cluster_name,
        cluster.mean_silhouette,
        anomaly.expected_revenue_stl,
        anomaly.residual_z,
        anomaly.is_anomaly,
        anomaly.anomaly_type,
        forecast.predicted_revenue,
        forecast.forecast_error,
        forecast.forecast_error_pct,
        forecast.below_expected_flag
    FROM ml.reseller_territory_monthly_metrics AS monthly
    LEFT JOIN ml.reseller_territory_cluster_result AS cluster
      ON monthly.territory_id = cluster.territory_id
    LEFT JOIN ml.reseller_territory_anomaly_monthly AS anomaly
      ON monthly.month_key = anomaly.month_key
     AND monthly.territory_id = anomaly.territory_id
    LEFT JOIN ml.reseller_territory_forecast_evaluation AS forecast
      ON monthly.month_key = forecast.month_key
     AND monthly.territory_id = forecast.territory_id
    '''

    with engine.begin() as connection:
        connection.execute(text(dashboard_view_sql))

    print(
        "Đã ghi kết quả Reseller vào schema ml và tạo "
        "ml.vw_reseller_territory_analysis_dashboard."
    )
else:
    print(
        "WRITE_TO_DATABASE=False: chưa ghi dữ liệu Reseller. "
        "Sau khi kiểm tra kết quả, đổi thành True và chạy lại cell này."
    )
